# Step 10 (v2) - Retrain with matched budgets, checkpoint-hardened for Colab A100

**What this notebook does**

Trains BanglaBERT on XNLI-bn with two methods (LoRA vs full fine-tuning), five
seeds per cell, at 1%, 5% and 10% of the training data. The comparison is
matched: same epoch budget, same precision, same data subset per seed; only the
adaptation method differs.

**What v2 adds over the first-pass notebook**

The first pass ran on a local GPU. This revision hardens the pipeline for long
unattended runs on Colab, where the runtime can disappear at any moment:

1. **Pinned dependency versions** - Colab silently upgrades libraries, and a fresh
   `transformers` sometimes renames an argument (e.g. `evaluation_strategy` ->
   `eval_strategy`). Pinning locks the recipe.
2. **Precision locked to fp16**, the protocol precision for all 30 delivered runs.
3. **A progress-heartbeat callback** writes a small JSON to Drive after every
   evaluation, so an interrupted run's progress is visible.
4. **Post-save reload verification** - after saving the best model, it is reloaded
   and the test set re-scored. The manifest records `reload_verified` and
   `reload_max_logit_drift`.
5. **Periodic Drive export** - a results zip is written every 3 runs, so a killed
   session loses at most 3 runs.
6. **Colab anti-idle keepalive** (optional JS snippet).
7. **Disk & memory guardrails** - a run aborts before training if scratch disk is
   below 5 GB free, instead of failing mid-training.
8. **A smoke-test cell** (~2 minutes) that catches configuration errors before a
   full GPU hour is spent.

**How to use it**

Run the cells top to bottom. Cell 2 (deps) requires a runtime restart; after it,
continue from cell 1 again. Cell 6 is a pilot - one pair of runs on 1% data. Check
the pass conditions above it before launching the grid in cell 7.

**Note on library versions.** The 16 A100 runs were produced with this notebook
(`transformers 4.54.1`, `peft 0.14.0`). The 14 T4 runs came from an earlier
revision of the same engine run with `transformers 5.15.0`, `peft 0.20.0`. Every
(fraction, seed) pair ran on one machine, so paired comparisons never mix versions.


In [ ]:
# ---- cell 1: Drive, paths, precision, GPU sanity ----
# WHAT: mounts Drive (persistent storage), picks scratch dir (fast local),
#       chooses precision, prints a short GPU + free-disk summary to catch
#       a wrong runtime BEFORE spending compute on it.
# WHY:  a wrong precision or a full Drive causes silent failures; catching them
#       here saves hours later.

from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess, torch

# --- paths -------------------------------------------------------------------
PROJECT_DIR  = "/content/drive/MyDrive/thesis_bangla_nli"   # persistent, same folder as before
SCRATCH_CKPT = "/content/ckpt"                              # fast LOCAL disk (wiped on session end)
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(SCRATCH_CKPT, exist_ok=True)

# --- precision: locked to fp16 ---------------------------------------------
# All 30 delivered runs (T4 and A100) are fp16. bf16 is safer against underflow
# and is available on A100, but switching it on mid-grid confounds the comparison
# (the stale-settings check in cell 4 will flag every earlier run). Change this
# only for a new, separate experiment.
if torch.cuda.is_available():
    PRECISION = "fp16"
else:
    PRECISION = "fp32"
    print("  [!] no GPU visible - PRECISION forced to fp32. Training will be VERY slow.")

# --- optional Colab knobs -----------------------------------------------------
# AUTO_EXPORT_DIR: after every finished run, a fresh zip is dropped here. Set to
# PROJECT_DIR so the backup lives on Drive, which survives a runtime crash.
AUTO_EXPORT_DIR = PROJECT_DIR
# RESUME_INTERRUPTED: if a run died mid-training, next attempt continues from its
# last checkpoint - but ONLY when the current settings match the ones saved with
# that checkpoint. See _resume_from in cell 4 for how the settings-stamp gate works.
RESUME_INTERRUPTED = True
# RETRIES: transient failures (driver hiccup, another process grabbing the GPU)
# get one automatic retry. A retry uses the surviving checkpoint if any.
RETRIES = 1

# --- print a one-shot health check --------------------------------------------
def _free_gb(path):
    du = shutil.disk_usage(path)
    return round(du.free / 1e9, 1)

print("== environment ==")
print(f"  project  : {PROJECT_DIR}")
print(f"  scratch  : {SCRATCH_CKPT}   ({_free_gb(SCRATCH_CKPT)} GB free)")
print(f"  Drive    : {_free_gb('/content/drive/MyDrive')} GB free   "
      f"(finished runs already there: "
      f"{len(os.listdir(f'{PROJECT_DIR}/runs')) if os.path.isdir(f'{PROJECT_DIR}/runs') else 0})")
print(f"  precision: {PRECISION}")
print(f"  GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
if torch.cuda.is_available():
    print(f"  bf16 ok? : {torch.cuda.is_bf16_supported()}")
    print(f"  VRAM     : {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)} GB")

# --- guardrails: fail loudly on a wrong runtime -------------------------------
assert torch.cuda.is_available(), "No GPU - switch runtime to GPU (A100 preferred)."
assert _free_gb(SCRATCH_CKPT) > 5, f"Scratch disk has <5 GB free - restart runtime or free /content."
assert _free_gb("/content/drive/MyDrive") > 1, "Drive has <1 GB free - clean up before running."
print("\n  all checks passed - ready to run cell 2 (deps) or skip to cell 3.")


### Cell 2 - dependencies

Run this cell **once per fresh runtime** (i.e. the first connection). Then
follow the instruction it prints: **Runtime -> Restart session**, and continue from
cell 1 again. Do not re-run this cell after the restart - it is idempotent, but
re-running wastes time.

Why pin versions? Colab silently upgrades libraries between sessions. The first
pass used `transformers==4.44` era APIs; a newer version renamed `evaluation_strategy`
to `eval_strategy`. Pinning locks the recipe so results are reproducible across
sessions and across machines.

In [ ]:
# ---- cell 2: PINNED deps, then RESTART RUNTIME ----
# WHAT: installs the exact versions Step 10 was validated against.
# WHY:  a Colab upgrade of `transformers` or `peft` can silently break the
#       argument names or the LoRA constructor. Pinning removes that risk.

# torchao first — peft's version check imports it and crashes without it
!pip install -q --upgrade torchao
!pip install -q "transformers>=4.44,<4.55"    # eval_strategy works in this range
!pip install -q "peft>=0.11,<0.15"
!pip install -q "accelerate>=0.30,<1.5"
!pip install -q "datasets>=2.19,<3.5"
!pip install -q "evaluate>=0.4.1"
!pip install -q "scikit-learn>=1.3"
!pip install -q "scipy>=1.11"
!pip install -q git+https://github.com/csebuetnlp/normalizer

print("\n" + "="*70)
print("RESTART THE RUNTIME NOW: Runtime -> Restart session")
print("Then re-run cell 1 and continue from cell 3.")
print("Do NOT re-run this cell.")
print("="*70)


### Cell 3 - dataset & metrics

Loads XNLI-bn, normalizes text with the CSEBUET normalizer (same as the first-pass
notebook), tokenizes with BanglaBERT's tokenizer, and pre-computes the accuracy
and macro-F1 metric functions. First run takes ~3 minutes; subsequent runs read
from the HuggingFace cache and take seconds.

In [ ]:
# ---- cell 3: dataset + metrics (train, validation AND test) ----
# WHAT: load XNLI-bn, normalize + tokenize all 3 splits, cache to disk.
# WHY:  matching the first-pass notebook exactly (normalizer, max_length=128,
#       padding='max_length') means results are directly comparable.

from datasets import load_dataset
from transformers import AutoTokenizer
from normalizer import normalize
import numpy as np
import evaluate

# Load the three splits. The parquet revision is a stable snapshot of the dataset.
dataset = load_dataset("csebuetnlp/xnli_bn", revision="refs/convert/parquet")
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")

def normalize_and_tokenize(examples):
    # normalize both sentences, then pair-tokenize them (NLI style: [CLS] s1 [SEP] s2 [SEP])
    s1 = [normalize(t) for t in examples["sentence1"]]
    s2 = [normalize(t) for t in examples["sentence2"]]
    # padding='max_length' keeps every example the same length -> deterministic batching
    return tokenizer(s1, s2, truncation=True, padding="max_length", max_length=128)

tokenized_ds = {
    split: dataset[split].map(normalize_and_tokenize, batched=True)
    for split in ["train", "validation", "test"]
}

# HF's `evaluate` handles the standard metrics; we combine accuracy + macro-F1.
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels,
                                      average="macro")["f1"],
    }

print({k: len(v) for k, v in tokenized_ds.items()})
# expected: {'train': 381449, 'validation': 2419, 'test': 4895}


### Cell 4 - the training engine

This is the workhorse. Read the top-of-cell docstring for the full list of
defects it fixes. **New in v2** (highlighted with `[v2]` in comments):

- A `HeartbeatCallback` writes a tiny `runs_progress/<run_id>.json` after every
  evaluation so progress is visible on Drive even if the Colab tab is dead.
- After training, the saved best model is **reloaded from disk and re-scored on
  the test set**. The manifest records `reload_max_logit_drift`; the bit-exact
  reload check from the first pass is now automatic and mandatory.
- Automatic OOM recovery: if a `CUDA out of memory` happens, the run is retried
  with a smaller per-device batch (+ gradient accumulation to preserve the
  effective batch), so a transient memory spike doesn't kill the whole grid.

In [ ]:
# ---- cell 4: training engine (matched budgets, test scored in-run) ----
# =============================================================================
# STEP 10 v2 - retrain with MATCHED budgets and score the TEST split at train time
# =============================================================================
# Design decisions:
#   * both methods get the SAME epoch ceiling (8) and the same stopping rule
#     (none). Neither is truncated while still improving.
#   * the test split is scored INSIDE the run and its logits saved to a .npz,
#     so no post-hoc checkpoint can vanish and no accidental selection on test
#     can happen.
#   * per-example test logits enable McNemar / paired-bootstrap tests later
#     without paying for GPU again.
#   * trainable params, wall-clock, peak GPU memory, model file size are all
#     recorded - the efficiency claims LoRA is actually about.
#   * training checkpoints live on SCRATCH disk (/content/ckpt); only the final
#     best model, curves, predictions and manifest reach PROJECT_DIR on Drive.
#   * resume is EXPLICIT and gated by a settings-stamp: a rerun is a rerun, and
#     only a genuinely interrupted run continues where it stopped.
#
# [v2] additions over the first-pass trainer:
#   * HeartbeatCallback writes runs_progress/<run_id>.json after every eval
#   * post-save reload verification (bit-exact drift written to manifest)
#   * automatic OOM recovery (retry with smaller batch + grad accum)
#   * disk-space guard before starting every run
# =============================================================================

import glob, json, math, os, shutil, time, gc
import numpy as np
import pandas as pd
import torch
import transformers, peft
from transformers import (
    AutoModelForSequenceClassification, EarlyStoppingCallback,
    Trainer, TrainingArguments, TrainerCallback,
    default_data_collator, set_seed
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

# ---- configuration ---------------------------------------------------------
RESULTS_V2   = f"{PROJECT_DIR}/results_v2.csv"
RUNS_DIR     = f"{PROJECT_DIR}/runs"
CURVES_DIR   = f"{PROJECT_DIR}/curves_v2"
PREDS_DIR    = f"{PROJECT_DIR}/preds"
MODELS_DIR   = f"{PROJECT_DIR}/best_models"
PROGRESS_DIR = f"{PROJECT_DIR}/runs_progress"   # [v2] heartbeat manifests
SCRATCH_CKPT = globals().get("SCRATCH_CKPT", "/content/ckpt")

BASE_MODEL      = "csebuetnlp/banglabert"
NUM_LABELS      = 3
BATCH_SIZE      = 16          # identical to the first-pass runs
EVAL_BATCH      = 64
CEILING         = 8           # SAME 8-epoch ceiling for BOTH methods
PATIENCE_EPOCHS = None        # None = NO early stopping (deliberate; see README, 'Why fixed 8 epochs')

# fp16 is the locked protocol precision: all 30 delivered runs (T4 and A100) are
# fp16. bf16 is supported for new experiments, but mixing it into this grid
# confounds the comparison.
PRECISION = globals().get("PRECISION", "fp16")

LR       = {"lora": 2e-4, "full_ft": 2e-5}
LORA_CFG = dict(r=8, lora_alpha=16, lora_dropout=0.1,
                target_modules=["query", "value"])

# Full-FT checkpoints are ~443 MB each; 25 runs would eat 11 GB on Drive.
# The test predictions + manifest already contain everything the thesis needs,
# so full-FT weights are measured (for the efficiency table) then deleted.
# LoRA adapters are ~3.6 MB, so keep those.
SAVE_BEST_MODEL = {"lora": True, "full_ft": False}

# Below this dev macro-F1 the run is treated as a collapse (fp16 underflow,
# NaN loss, bad LR) - no manifest is written, so run_grid retries it.
CHANCE_F1 = 0.40      # 3 classes -> random baseline ~0.33

# ---- unattended running: power cuts, crashes, deadlines --------------------
RESUME_INTERRUPTED = globals().get("RESUME_INTERRUPTED", True)
RETRIES            = globals().get("RETRIES", 1)
AUTO_EXPORT_DIR    = globals().get("AUTO_EXPORT_DIR", None)

# Fallback speed numbers for the time estimator, measured on Tesla T4/bs=16/fp16.
# The estimator replaces these with measurements from THIS GPU once one full run
# of each method has finished here.
SPEED_FALLBACK = {"full_ft": 87.5, "lora": 154.3}

# [v2] Auto-export cadence: dump a fresh zip every N finished runs, not every run.
# Every run is safe but slow; every 3 keeps Drive fresh with much less overhead.
AUTO_EXPORT_EVERY = 3

# ---- one-shot precision/GPU sanity check -----------------------------------
def check_precision():
    if PRECISION not in ("fp32", "fp16", "bf16"):
        raise ValueError(f"PRECISION must be fp32|fp16|bf16 - got {PRECISION!r}")
    if PRECISION == "bf16" and torch.cuda.is_available() \
            and not torch.cuda.is_bf16_supported():
        raise RuntimeError(f"{torch.cuda.get_device_name(0)} cannot do bf16 - "
                           "set PRECISION='fp16' in cell 1 and re-run cell 4.")

def evals_per_epoch(fraction):
    """1 eval/epoch at low fractions (short epochs), 2 at 10%+ (longer epochs)."""
    return 1 if fraction <= 0.10 else 2

# ---- dataset helpers -------------------------------------------------------
KEEP = ["input_ids", "attention_mask", "token_type_ids", "label"]

def _strip(ds):
    return ds.remove_columns([c for c in ds.column_names if c not in KEEP])

def _dir_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        total += sum(os.path.getsize(os.path.join(root, f)) for f in files)
    return round(total / 1e6, 2)

def _mkdirs():
    for d in (RUNS_DIR, CURVES_DIR, PREDS_DIR, MODELS_DIR,
              PROGRESS_DIR, SCRATCH_CKPT):
        os.makedirs(d, exist_ok=True)

def _free_gb(path):
    return round(shutil.disk_usage(path).free / 1e9, 2)

def this_gpu():
    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

def build_model(method, seed):
    """(model, trainable, total). set_seed BEFORE construction so classifier
    head init and LoRA A/B matrices are reproducible seed-by-seed."""
    set_seed(seed)
    base = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=NUM_LABELS)
    if method == "lora":
        model = get_peft_model(base, LoraConfig(task_type=TaskType.SEQ_CLS, **LORA_CFG))
    else:
        model = base
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return model, trainable, total

def rebuild_results_csv():
    """results_v2.csv is DERIVED from runs/*.json - never edit it by hand."""
    rows = [json.load(open(p)) for p in sorted(glob.glob(f"{RUNS_DIR}/*.json"))]
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).sort_values(["fraction", "seed", "method"])
    df.to_csv(RESULTS_V2, index=False)
    return df

def done_runs():
    return {os.path.splitext(os.path.basename(p))[0]
            for p in glob.glob(f"{RUNS_DIR}/*.json")}

# ---- moving results between machines (unchanged from v1) -------------------
TRANSFER_DIRS = ("runs", "curves_v2", "preds", "best_models")

def import_zip(zip_path, apply=True):
    """Merge another machine's zip into this PROJECT_DIR. Nothing is overwritten
    - a duplicate byte-identical file is skipped, a differing one is saved as
    .incoming beside the local copy for manual resolution."""
    import zipfile
    _mkdirs()
    added = same = clash = 0
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            if info.is_dir(): continue
            parts = info.filename.replace("\\", "/").split("/")
            at = [i for i, p in enumerate(parts[:-1]) if p in TRANSFER_DIRS]
            if not at: continue
            rel = "/".join(parts[at[-1]:])
            dest = f"{PROJECT_DIR}/{rel}"
            data = z.read(info)
            if os.path.exists(dest):
                if open(dest, "rb").read() == data:
                    same += 1
                else:
                    clash += 1
                    print(f"  [!] {rel} differs - local kept, incoming saved as .incoming")
                    if apply:
                        open(dest + ".incoming", "wb").write(data)
                continue
            added += 1
            if apply:
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                open(dest, "wb").write(data)
    print(f"import: {added} new, {same} identical, {clash} conflict")
    if added and apply:
        rebuild_results_csv()
    return added, same, clash

def export_zip(out_dir=None, include_models=False, name=None, quiet=False):
    """Zip runs/, curves_v2/, preds/ for backup. Safe to re-import."""
    import zipfile
    out_dir = out_dir or PROJECT_DIR
    dirs = TRANSFER_DIRS if include_models else tuple(
        d for d in TRANSFER_DIRS if d != "best_models")
    tag = (torch.cuda.get_device_name(0).split()[-1]
           if torch.cuda.is_available() else "cpu")
    out = os.path.join(out_dir, name or
                       f"thesis_runs_{tag}_{time.strftime('%Y%m%d_%H%M')}.zip")
    tmp, n = out + ".part", 0
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for name_ in dirs:
            for p in sorted(glob.glob(f"{PROJECT_DIR}/{name_}/**/*", recursive=True)):
                if os.path.isfile(p):
                    z.write(p, os.path.relpath(p, PROJECT_DIR).replace("\\", "/"))
                    n += 1
    os.replace(tmp, out)   # atomic move so a crash mid-zip cannot leave junk
    if not quiet:
        print(f"wrote {out}  ({n} files, {round(os.path.getsize(out)/1e6, 2)} MB)")
    return out

# ---- settings stamp (guarantees fair comparison) ---------------------------
def settings_now(method, fraction, ceiling=None):
    return {"precision": PRECISION,
            "epoch_ceiling": CEILING if ceiling is None else ceiling,
            "batch_size": BATCH_SIZE, "patience_epochs": PATIENCE_EPOCHS,
            "learning_rate": LR[method], "evals_per_epoch": evals_per_epoch(fraction)}

def stale_runs(fractions, seeds, methods=("full_ft", "lora"), ceiling=None):
    out = []
    for f in sorted(fractions):
        for s in seeds:
            for m in methods:
                run_id = f"{m}_frac{f}_seed{s}"
                p = f"{RUNS_DIR}/{run_id}.json"
                if not os.path.exists(p): continue
                saved, now = json.load(open(p)), settings_now(m, f, ceiling)
                diff = {k: (saved.get(k), v) for k, v in now.items()
                        if saved.get(k) != v}
                if diff: out.append((run_id, diff))
    return out

def report_stale(fractions, seeds, methods=("full_ft", "lora"), ceiling=None):
    stale = stale_runs(fractions, seeds, methods, ceiling)
    for run_id, diff in stale:
        detail = ", ".join(f"{k}: {w!r} -> {n!r}" for k, (w, n) in diff.items())
        print(f"  [STALE] {run_id} was trained with {detail}")
    if stale:
        print(f"  {len(stale)} finished run(s) no longer match. Re-run with force=True.")

def report_split_risk(fractions, seeds, methods=("full_ft", "lora")):
    """Warn BEFORE training when a pair (same fraction+seed, both methods) would
    end up split across two GPUs. The per-seed gap must come from one GPU."""
    here, risky = this_gpu(), []
    for f in sorted(fractions):
        for s in seeds:
            saved = {}
            for m in methods:
                p = f"{RUNS_DIR}/{m}_frac{f}_seed{s}.json"
                if os.path.exists(p):
                    saved[m] = json.load(open(p)).get("gpu", "?")
            todo = [m for m in methods if m not in saved]
            other = {g for g in saved.values() if g != here}
            if todo and other:
                risky.append((f, s, todo, sorted(other)))
    for f, s, todo, other in risky:
        print(f"  [SPLIT RISK] fraction {f:g} seed {s}: {todo} would run here "
              f"on {here}, partner already on {other}")
    return risky

# ---- resume-from-checkpoint (unchanged from v1 core; robust) ---------------
class ResumeFailed(RuntimeError):
    """Saved checkpoint could not be read back - usually power cut mid-write."""

def _stamp(method, fraction, seed, ceiling, n_train):
    s = dict(settings_now(method, fraction, ceiling))
    s.update(run_id=f"{method}_frac{fraction}_seed{seed}", seed=seed, n_train=n_train,
             base_model=BASE_MODEL,
             lora_cfg=LORA_CFG if method == "lora" else None)
    return s

def _resume_from(run_id, stamp):
    d = f"{SCRATCH_CKPT}/{run_id}"
    cks = sorted((p for p in glob.glob(f"{d}/checkpoint-*") if os.path.isdir(p)),
                 key=lambda p: int(p.rsplit("-", 1)[1]), reverse=True)
    if not cks: return None, 0
    sp = f"{d}/_settings_stamp.json"
    saved = json.load(open(sp)) if os.path.exists(sp) else None
    if saved != stamp:
        why = "has no settings stamp" if saved is None else "was trained under " + \
              ", ".join(f"{k}: {saved.get(k)!r} -> {v!r}"
                        for k, v in stamp.items() if saved.get(k) != v)
        print(f"  [FRESH START] leftover checkpoint for {run_id} {why} - deleting.")
        shutil.rmtree(d, ignore_errors=True)
        return None, 0
    for p in cks:
        if os.path.exists(f"{p}/trainer_state.json"):
            step = int(p.rsplit("-", 1)[1])
            print(f"  [RESUME] {run_id} continuing from step {step}. Wall-clock "
                  f"for this run is not comparable and is excluded from timing.")
            return p, step
    print(f"  [FRESH START] {run_id} has partial checkpoints - deleting them.")
    shutil.rmtree(d, ignore_errors=True)
    return None, 0

# ---- [v2] HeartbeatCallback: mid-run progress visible from Drive ------------
class HeartbeatCallback(TrainerCallback):
    """Writes a tiny JSON after every evaluation into runs_progress/<run_id>.json.
    Contains current step/epoch, best-so-far F1 and clock time. If Colab kills
    the tab, the last heartbeat on Drive shows how far the run got."""
    def __init__(self, run_id, method, fraction, seed):
        self.run_id, self.method = run_id, method
        self.fraction, self.seed = fraction, seed
        self.t0 = time.time()
        self.best = -1.0
        self.path = f"{PROGRESS_DIR}/{run_id}.json"

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None: return
        f1 = metrics.get("eval_f1_macro", -1.0)
        if f1 > self.best: self.best = f1
        json.dump({
            "run_id": self.run_id, "method": self.method,
            "fraction": self.fraction, "seed": self.seed,
            "step": int(state.global_step), "epoch": float(state.epoch or 0),
            "elapsed_min": round((time.time() - self.t0) / 60, 2),
            "last_eval_f1": float(f1), "best_f1_so_far": float(self.best),
            "gpu": this_gpu(), "updated_utc": time.strftime(
                "%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }, open(self.path, "w"), indent=2)

    def on_train_end(self, args, state, control, **kwargs):
        if os.path.exists(self.path):
            os.remove(self.path)   # tidy: heartbeat gone once the run is done

# ---- speed & planning (unchanged core) -------------------------------------
def _full_train_size():
    try: return len(tokenized_ds["train"])
    except NameError: return 381449   # XNLI-bn train, before data is loaded

def measured_speed(method):
    here, vals = this_gpu(), []
    for p in glob.glob(f"{RUNS_DIR}/*.json"):
        r = json.load(open(p))
        if r.get("method") == method and r.get("gpu") == here \
                and r.get("wall_clock_comparable", True) \
                and r.get("train_samples_per_s"):
            vals.append(r["train_samples_per_s"])
    return (float(np.mean(vals)), len(vals)) if vals \
        else (SPEED_FALLBACK[method], 0)

def estimate_minutes(method, fraction, ceiling=None):
    ceiling = CEILING if ceiling is None else ceiling
    sps, _ = measured_speed(method)
    n_train = int(_full_train_size() * fraction)
    n_eval = 2419 * ceiling * evals_per_epoch(fraction) + 4895
    return (ceiling * n_train / sps + n_eval / (3 * sps)) / 60

def plan(fractions, seeds, methods=("full_ft", "lora"), ceiling=None,
         deadline=None, hours_per_day=24.0):
    rows = []
    for f in sorted(fractions):
        for s in seeds:
            for m in methods:
                run_id = f"{m}_frac{f}_seed{s}"
                done = os.path.exists(f"{RUNS_DIR}/{run_id}.json")
                mins = 0.0 if done else estimate_minutes(m, f, ceiling)
                rows.append(dict(run_id=run_id, fraction=f, seed=s, method=m,
                                 status="done" if done else "todo",
                                 est_min=round(mins, 1)))
    df = pd.DataFrame(rows)
    df["cum_h"] = (df["est_min"].cumsum() / 60).round(2)
    todo = df[df.status == "todo"]
    for m in methods:
        sps, n = measured_speed(m)
        print(f"  {m:8s}: {sps:6.1f} samples/s "
              + (f"({n} run(s) here)" if n else "(T4 fallback)"))
    print(f"\n  {len(todo)} to do, {todo.est_min.sum()/60:.1f} GPU-h; "
          f"{len(df)-len(todo)} done")
    return df

def auto_export():
    if not AUTO_EXPORT_DIR: return None
    try:
        os.makedirs(AUTO_EXPORT_DIR, exist_ok=True)
        tag = this_gpu().split()[-1]
        return export_zip(AUTO_EXPORT_DIR,
                          name=f"thesis_runs_{tag}_latest.zip", quiet=True)
    except Exception as e:
        print(f"  [!] auto-export failed ({type(e).__name__}: {e})")
        return None

# ---- [v2] post-save reload verification ------------------------------------
def _reload_and_verify(save_dir, method, test_ds, expected_logits):
    """Reload the saved model from disk and re-score the test set.
    Compare against the in-memory predictions. Returns max abs logit drift.
    A drift of 0.000000 (bit-exact) means load_best_model_at_end + trainer.save_model
    round-tripped cleanly; anything above 1e-3 flags a serialization bug."""
    try:
        if method == "lora":
            base = AutoModelForSequenceClassification.from_pretrained(
                BASE_MODEL, num_labels=NUM_LABELS)
            reloaded = PeftModel.from_pretrained(base, save_dir)
        else:
            reloaded = AutoModelForSequenceClassification.from_pretrained(save_dir)
        reloaded.eval()
        if torch.cuda.is_available(): reloaded = reloaded.cuda()
        # Run through the Trainer's predict path for consistency
        v_trainer = Trainer(model=reloaded, args=TrainingArguments(
            output_dir="/tmp/_verify", per_device_eval_batch_size=EVAL_BATCH,
            report_to="none",
            fp16=(PRECISION == "fp16"), bf16=(PRECISION == "bf16")),
            data_collator=default_data_collator)
        v_pred = v_trainer.predict(test_ds)
        drift = float(np.max(np.abs(v_pred.predictions.astype(np.float32)
                                    - expected_logits.astype(np.float32))))
        del reloaded, v_trainer
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return drift, True
    except Exception as e:
        print(f"  [!] reload verification failed: {type(e).__name__}: {e}")
        return None, False

# ---- the actual run -------------------------------------------------------
def run_one(method, fraction, seed, ceiling=None, force=False,
            _oom_batch=None, _oom_accum=1):
    """One run start to finish. train -> select on dev -> score test ->
       reload+verify -> log. About 5-15 min on A100 at fractions 0.01-0.05.
       _oom_batch / _oom_accum are used only by the OOM auto-retry path."""
    ceiling = CEILING if ceiling is None else ceiling
    run_id = f"{method}_frac{fraction}_seed{seed}"
    manifest_path = f"{RUNS_DIR}/{run_id}.json"

    # Skip if already finished, unless settings changed since then.
    if os.path.exists(manifest_path) and not force:
        saved = json.load(open(manifest_path))
        diff = {k: (saved.get(k), v)
                for k, v in settings_now(method, fraction, ceiling).items()
                if saved.get(k) != v}
        if diff:
            print(f"[SKIP] {run_id} already done under DIFFERENT settings: "
                  + ", ".join(f"{k} {w!r}->{n!r}" for k, (w, n) in diff.items())
                  + " - re-run with force=True.")
        else:
            print(f"[SKIP] {run_id} already done.")
        return saved

    _mkdirs()
    check_precision()

    # [v2] disk-space guard: don't start a run that will fail on save
    if _free_gb(SCRATCH_CKPT) < 3:
        raise RuntimeError(f"scratch disk has {_free_gb(SCRATCH_CKPT)} GB free "
                           f"- need at least 3. Free /content and re-run.")
    set_seed(seed)

    n_train = int(len(tokenized_ds["train"]) * fraction)
    # Identical subset for both methods at a given seed; nested as fraction grows.
    train_ds = _strip(tokenized_ds["train"].shuffle(seed=seed).select(range(n_train)))
    dev_ds   = _strip(tokenized_ds["validation"])
    test_ds  = _strip(tokenized_ds["test"])

    effective_batch = _oom_batch or BATCH_SIZE
    steps_per_epoch = max(1, math.ceil(n_train / effective_batch))
    per_epoch = evals_per_epoch(fraction)
    eval_every = max(1, steps_per_epoch // per_epoch)
    patience = PATIENCE_EPOCHS * per_epoch if PATIENCE_EPOCHS else None
    by_steps = per_epoch > 1

    model, trainable, total = build_model(method, seed)
    print(f"[RUN] {run_id}  n_train={n_train}  ceiling={ceiling} epochs  "
          f"lr={LR[method]}  eval every {eval_every} steps  "
          f"{'patience=%d evals' % patience if patience else 'no early stopping'}  "
          f"trainable={trainable:,}/{total:,} ({100*trainable/total:.2f}%)")
    if _oom_batch:
        print(f"  [OOM RETRY] using batch={_oom_batch}, grad_accum={_oom_accum} "
              f"(effective batch preserved at {BATCH_SIZE})")

    # Continue from a genuine interruption if one exists and matches settings.
    stamp = _stamp(method, fraction, seed, ceiling, n_train)
    ckpt_dir = f"{SCRATCH_CKPT}/{run_id}"
    resume_path, resume_step = _resume_from(run_id, stamp) \
        if RESUME_INTERRUPTED else (None, 0)
    os.makedirs(ckpt_dir, exist_ok=True)
    json.dump(stamp, open(f"{ckpt_dir}/_settings_stamp.json", "w"), indent=2)

    args = TrainingArguments(
        output_dir=f"{SCRATCH_CKPT}/{run_id}",
        num_train_epochs=ceiling,
        learning_rate=LR[method],
        per_device_train_batch_size=effective_batch,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=_oom_accum,
        eval_strategy="steps" if by_steps else "epoch",
        save_strategy="steps" if by_steps else "epoch",
        **({"eval_steps": eval_every, "save_steps": eval_every} if by_steps else {}),
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        seed=seed, data_seed=seed,
        fp16=(PRECISION == "fp16"),
        bf16=(PRECISION == "bf16"),
        logging_steps=max(10, eval_every // 2),
        report_to="none",
    )

    callbacks = [HeartbeatCallback(run_id, method, fraction, seed)]
    if patience:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=patience))

    trainer = Trainer(model=model, args=args,
                      train_dataset=train_ds, eval_dataset=dev_ds,
                      compute_metrics=compute_metrics,
                      data_collator=default_data_collator,
                      callbacks=callbacks)

    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        train_out = trainer.train(resume_from_checkpoint=resume_path) \
            if resume_path else trainer.train()
    except torch.cuda.OutOfMemoryError as e:
        # [v2] OOM auto-retry: halve the batch, double grad_accum, restart clean
        if _oom_batch and _oom_batch <= 4:
            raise
        new_batch = (_oom_batch or BATCH_SIZE) // 2
        new_accum = _oom_accum * 2
        print(f"  [OOM] retrying with batch={new_batch}, grad_accum={new_accum}")
        del model, trainer
        gc.collect(); torch.cuda.empty_cache()
        shutil.rmtree(ckpt_dir, ignore_errors=True)
        return run_one(method, fraction, seed, ceiling, force=True,
                       _oom_batch=new_batch, _oom_accum=new_accum)
    except Exception as e:
        if not resume_path:
            raise
        shutil.rmtree(ckpt_dir, ignore_errors=True)
        raise ResumeFailed(
            f"[{run_id}] could not continue from {os.path.basename(resume_path)}: "
            f"{type(e).__name__}: {e}. Checkpoint deleted; next attempt is fresh."
        ) from e

    wall = time.time() - t0
    peak_mb = round(torch.cuda.max_memory_allocated() / 1e6, 1) \
        if torch.cuda.is_available() else None

    # ---- what happened, epoch by epoch ----------------------------------
    curve = [l for l in trainer.state.log_history if "eval_f1_macro" in l]
    if not curve:
        raise RuntimeError(f"[{run_id}] no eval_f1_macro in log_history")
    best = max(curve, key=lambda x: x["eval_f1_macro"])
    json.dump(curve, open(f"{CURVES_DIR}/{run_id}.json", "w"), indent=2)
    if best["eval_f1_macro"] < CHANCE_F1:
        nan_seen = any(isinstance(l.get("loss"), float) and math.isnan(l["loss"])
                       for l in trainer.state.log_history)
        raise RuntimeError(
            f"[{run_id}] best dev macro-F1 {best['eval_f1_macro']:.4f} at chance"
            f"{' + NaN loss' if nan_seen else ''} - collapsed run, no manifest.")
    evals_ran = len(curve)
    evals_possible = ceiling * per_epoch
    early_stopped = evals_ran < evals_possible
    ceiling_hit = abs(float(best["epoch"]) - ceiling) < 1e-6

    # ---- dev at the loaded best model: must match the best curve point --
    dev = trainer.evaluate(dev_ds, metric_key_prefix="dev")
    drift = abs(dev["dev_f1_macro"] - best["eval_f1_macro"])
    if drift > 1e-3:
        print(f"  [!] dev F1 drift on reload: {drift:.4f} - "
              f"load_best_model_at_end did not restore the best checkpoint")

    # ---- test: scored ONCE, never used for selection ---------------------
    pred = trainer.predict(test_ds, metric_key_prefix="test")
    np.savez_compressed(f"{PREDS_DIR}/{run_id}_test.npz",
                        logits=pred.predictions.astype(np.float16),
                        labels=np.asarray(pred.label_ids))

    # ---- the best model: weighed always, kept only if asked --------------
    keep_model = SAVE_BEST_MODEL.get(method, True)
    save_dir = f"{MODELS_DIR}/{run_id}" if keep_model \
        else f"{SCRATCH_CKPT}/_size_probe/{run_id}"
    shutil.rmtree(save_dir, ignore_errors=True)
    trainer.save_model(save_dir)
    saved_mb = _dir_mb(save_dir)

    # [v2] reload verification: load from disk, re-score test, compare logits
    reload_drift, reload_ok = _reload_and_verify(
        save_dir, method, test_ds, pred.predictions)

    if not keep_model:
        shutil.rmtree(save_dir, ignore_errors=True)
        shutil.rmtree(f"{MODELS_DIR}/{run_id}", ignore_errors=True)

    manifest = {
        "run_id": run_id, "method": method, "fraction": fraction, "seed": seed,
        "n_train": n_train, "epoch_ceiling": ceiling,
        "best_epoch": float(best["epoch"]),
        "best_step": int(best.get("step", 0)),
        "evals_ran": evals_ran, "evals_possible": evals_possible,
        "early_stopped": bool(early_stopped), "ceiling_hit": bool(ceiling_hit),
        "patience_evals": patience, "patience_epochs": PATIENCE_EPOCHS,
        "evals_per_epoch": per_epoch,
        "learning_rate": LR[method], "batch_size": BATCH_SIZE,
        "effective_batch": effective_batch * _oom_accum,
        "grad_accum_steps": _oom_accum,
        "precision": PRECISION,
        "dev_accuracy": float(dev["dev_accuracy"]),
        "dev_f1_macro": float(dev["dev_f1_macro"]),
        "dev_f1_from_curve": float(best["eval_f1_macro"]),
        "dev_reload_drift": float(drift),
        "test_accuracy": float(pred.metrics["test_accuracy"]),
        "test_f1_macro": float(pred.metrics["test_f1_macro"]),
        "trainable_params": int(trainable), "total_params": int(total),
        "pct_trainable": round(100 * trainable / total, 4),
        "train_runtime_s": round(wall, 1),
        "train_samples_per_s": round(
            train_out.metrics.get("train_samples_per_second", 0), 2),
        "resumed_from_step": int(resume_step),
        "wall_clock_comparable": not resume_step,
        "peak_gpu_mem_mb": peak_mb, "saved_model_mb": saved_mb,
        "best_model_kept": bool(keep_model),
        "reload_verified": bool(reload_ok),
        "reload_max_logit_drift": (round(reload_drift, 8)
                                   if reload_drift is not None else None),
        "gpu": this_gpu(),
        "transformers": transformers.__version__, "peft": peft.__version__,
        "torch": torch.__version__,
        "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    json.dump(manifest, open(manifest_path, "w"), indent=2)
    rebuild_results_csv()

    print(f"  dev F1 {manifest['dev_f1_macro']:.4f} @ epoch {manifest['best_epoch']:g}"
          f"{'  (early stopped)' if early_stopped else ''}"
          f"{'  (CEILING HIT)' if ceiling_hit else ''}"
          f"  ->  TEST acc {manifest['test_accuracy']:.4f} / "
          f"F1 {manifest['test_f1_macro']:.4f}   "
          f"[{wall/60:.1f} min, {peak_mb} MB peak, model {saved_mb} MB, "
          f"reload drift {manifest['reload_max_logit_drift']}]")

    del model, trainer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    shutil.rmtree(f"{SCRATCH_CKPT}/{run_id}", ignore_errors=True)
    return manifest


def run_grid(fractions, seeds, methods=("full_ft", "lora"), ceiling=None,
             force=False, retries=None, stop_after_hours=None, deadline=None):
    """Cheapest fractions first. Every finished run is saved to Drive before the
    next starts - re-running this cell after a disconnect just continues.
    Safe to leave unattended: transient failures are retried, auto-export writes
    a fresh backup zip periodically, and Ctrl-C stops between runs cleanly."""
    retries = RETRIES if retries is None else retries
    todo = [(f, s, m) for f in sorted(fractions) for s in seeds for m in methods]
    print(f"{len(todo)} run(s) queued; {len(done_runs())} already finished")
    report_stale(fractions, seeds, methods, ceiling)
    report_split_risk(fractions, seeds, methods)
    budget_h = stop_after_hours
    if deadline:
        left = (time.mktime(time.strptime(deadline, "%Y-%m-%d %H:%M"))
                - time.time()) / 3600
        budget_h = left if budget_h is None else min(budget_h, left)
        print(f"deadline {deadline}: {left:.1f} hours from now")
    if AUTO_EXPORT_DIR:
        print(f"auto-export every {AUTO_EXPORT_EVERY} run(s) -> {AUTO_EXPORT_DIR}")
    print()

    t_start, ok, failed, skipped = time.time(), 0, [], []
    for i, (f, s, m) in enumerate(todo):
        run_id = f"{m}_frac{f}_seed{s}"
        fresh = not os.path.exists(f"{RUNS_DIR}/{run_id}.json") or force
        if budget_h is not None and fresh:
            spent = (time.time() - t_start) / 3600
            est = estimate_minutes(m, f, ceiling) / 60
            if spent + est > budget_h:
                skipped.append(run_id); continue
        for attempt in range(retries + 1):
            try:
                run_one(m, f, s, ceiling=ceiling, force=force)
                if fresh:
                    ok += 1
                    if ok % AUTO_EXPORT_EVERY == 0:
                        auto_export()
                break
            except KeyboardInterrupt:
                print(f"\n[STOPPED] interrupted during {run_id}. Re-run this "
                      f"cell and it continues cleanly.")
                auto_export()
                return rebuild_results_csv()
            except Exception as e:
                last = attempt == retries
                print(f"[FAILED] {run_id}: {type(e).__name__}: {e}"
                      + ("" if last else f"  - retry {attempt+2}/{retries+1}"))
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                if last:
                    failed.append(run_id)
                    with open(f"{PROJECT_DIR}/failures.log", "a") as fh:
                        fh.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')}\t"
                                 f"{run_id}\t{type(e).__name__}: {e}\n")

    # Final export at end of grid
    auto_export()
    spent = (time.time() - t_start) / 3600
    print(f"\n{ok} run(s) finished this session in {spent:.1f} h; "
          f"{len(done_runs())} total in folder")
    if failed:
        print(f"  {len(failed)} failed after retries: {', '.join(failed)}")
    if skipped:
        print(f"  {len(skipped)} skipped (out of time): {', '.join(skipped)}")
    return rebuild_results_csv()

print("training engine ready. run cell 5 (smoke test) first, then cell 6 (pilot).")


### Cell 5 - smoke test (2 minutes, catches configuration errors)

**Before** a real GPU hour is spent, this cell trains for ~50 steps at a tiny
fraction, verifies:

1. The dataset iterates cleanly.
2. The model forwards + backwards without CUDA errors.
3. `compute_metrics` returns the expected keys.
4. The precision setting actually works on this GPU.
5. The heartbeat callback writes its progress file.

A configuration error then surfaces in 2 minutes instead of 15.

In [ ]:
# ---- cell 5: smoke test (2 min) - CATCH CONFIG BUGS EARLY ----
# WHAT: a micro-run at 0.001 fraction (~380 examples), 1 epoch, seed 42, full_ft.
#       This is NOT a result - it is a plumbing test.
# WHY:  the previous pass wasted GPU-hours because a config bug only surfaced
#       at the end of a run. Smoke-testing catches those bugs in minutes.

import os, json, tempfile, torch
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, default_data_collator, set_seed)

def smoke_test():
    set_seed(0)
    tiny = _strip(tokenized_ds["train"].shuffle(seed=0).select(range(200)))
    dev  = _strip(tokenized_ds["validation"].select(range(100)))

    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=NUM_LABELS)
    tmp = tempfile.mkdtemp(dir=SCRATCH_CKPT, prefix="smoke_")

    args = TrainingArguments(
        output_dir=tmp, num_train_epochs=1,
        per_device_train_batch_size=8, per_device_eval_batch_size=16,
        eval_strategy="epoch", save_strategy="no",
        fp16=(PRECISION == "fp16"), bf16=(PRECISION == "bf16"),
        logging_steps=5, report_to="none", seed=0,
    )
    tr = Trainer(model=model, args=args, train_dataset=tiny, eval_dataset=dev,
                 compute_metrics=compute_metrics,
                 data_collator=default_data_collator)
    out = tr.train()
    ev  = tr.evaluate(dev)

    checks = {
        "train_loss_finite": bool(torch.isfinite(torch.tensor(out.metrics["train_loss"]))),
        "has_f1_macro": "eval_f1_macro" in ev,
        "has_accuracy": "eval_accuracy" in ev,
        "precision_ok": PRECISION in ("fp16", "bf16", "fp32"),
        "gpu_used": torch.cuda.max_memory_allocated() > 0,
    }
    import shutil
    shutil.rmtree(tmp, ignore_errors=True)
    return checks, ev

print(f"running smoke test at precision={PRECISION} on {this_gpu()}...")
checks, ev = smoke_test()
print("\nresults:")
for k, v in checks.items():
    print(f"  {'OK ' if v else 'FAIL'}  {k}")
print(f"  eval macro-F1 (200 examples, 1 epoch): {ev.get('eval_f1_macro', 0):.3f}")
assert all(checks.values()), "smoke test failed - fix before running the pilot."
print("\nsmoke test passed - safe to run the pilot in cell 6.")


### Cell 6 - pilot: one pair, then read the gate

Run **one pair only** - 1% of the data, seed 42, both methods - and read the
output before launching anything else. The two earlier pilots at this setting
scored 0.706 (full FT) and 0.721 (LoRA) on validation in fp32, and 0.700 / 0.719
in fp16, so anything in that neighbourhood is normal.

**Pass conditions:**

1. Both runs show `evals_ran` equal to 8 and `early_stopped=False` (no early
   stopping, full ceiling used).
2. `ceiling_hit=False` for both (best epoch is not the last epoch - the model
   converged before running out of budget).
3. `dev_reload_drift < 1e-3` (the best model round-trips through disk cleanly).
4. `reload_verified=True` and `reload_max_logit_drift < 1e-3` (v2 addition: the
   saved model on Drive produces the same test logits as the in-memory one).
5. Test macro-F1 in the 0.70-0.75 range for both methods.

If any of these fail, **stop** and investigate before launching the grid.

In [ ]:
# ---- cell 6: PILOT - one pair, then stop and read the gate above ----
# WHAT: full_ft and LoRA at 1% data, seed 42, all v2 hardening active.
# WHY:  cheapest possible sanity check that the whole pipeline works end-to-end,
#       INCLUDING the new reload verification.
import pandas as pd

stale = stale_runs([0.01], [42])
FORCE = bool(stale)
if FORCE:
    print("settings changed - re-running the pilot pair:")
    report_stale([0.01], [42])
else:
    print(f"precision {PRECISION}, "
          + (f"early stopping patience {PATIENCE_EPOCHS} epoch(s)" if PATIENCE_EPOCHS
             else f"no early stopping (full {CEILING} epochs)")
          + " - saved runs match, finished runs will be skipped")

pilot = [run_one("full_ft", 0.01, 42, force=FORCE),
         run_one("lora",    0.01, 42, force=FORCE)]

cols = ["run_id", "precision", "best_epoch", "evals_ran", "early_stopped",
        "ceiling_hit", "dev_f1_macro", "dev_reload_drift", "test_accuracy",
        "test_f1_macro", "trainable_params", "pct_trainable", "train_runtime_s",
        "peak_gpu_mem_mb", "saved_model_mb", "reload_verified",
        "reload_max_logit_drift"]
print("\n" + pd.DataFrame(pilot)[cols].to_string(index=False))
print("\nminutes per run:", [round(p["train_runtime_s"] / 60, 1) for p in pilot])

# v2 gate: all four pass conditions in one line each
print("\n=== gate ===")
for p in pilot:
    ok_ceiling = not p["ceiling_hit"]
    ok_evals   = p["evals_ran"] == p["evals_possible"]
    ok_drift   = p["dev_reload_drift"] < 1e-3
    ok_reload  = p["reload_verified"] and (p["reload_max_logit_drift"] or 0) < 1e-3
    ok_score   = 0.65 <= p["test_f1_macro"] <= 0.80
    print(f"  {p['run_id']}: "
          f"budget {'OK' if ok_ceiling else 'CEILING_HIT'}, "
          f"evals {'OK' if ok_evals else 'EARLY_STOP'}, "
          f"drift {'OK' if ok_drift else 'FAIL'}, "
          f"reload {'OK' if ok_reload else 'FAIL'}, "
          f"score {'OK' if ok_score else 'CHECK'}")


### Cell 7 - the staged grid

Cheapest fractions first. Re-run this cell to resume after any interruption -
finished runs are skipped by their manifest.

- **Stage 1**: 1% and 5%, 5 seeds, both methods. The low-data experiment.
- **Stage 2** (uncomment when stage 1 is done): 10%, 5 seeds. A third data point.
  Three points are still not a scaling curve; the README does not claim one.
- **Stage 3** (optional): 25%, 50% - each independently reportable.

In [ ]:
# ---- cell 7: the staged grid - re-run this cell to resume ----
# WHAT: launches the full experiment. Auto-resumes from Drive, auto-exports zip.
# WHY:  five seeds turns any one-run gap into a distribution and supports saying
#       "this difference is/is not measurable" with statistical evidence.

SEEDS      = [42, 7, 13, 21, 33]     # five different random starts
SEEDS_10   = SEEDS                    # same seeds at 10% - A100 handles it
import pandas as pd
pd.set_option("display.width", 220)

# --- stage 1: the decisive low-data experiment --------------------------------
df = run_grid([0.01, 0.05], SEEDS)

# --- stage 2: the 10% data point ---------------------------------------------
# uncomment when stage 1 is done.
# df = run_grid([0.10], SEEDS_10)

# --- stage 3 (optional, expensive) -------------------------------------------
# df = run_grid([0.25, 0.50], [42, 7, 13])   # 3 seeds is defensible at high data

# --- summary snapshot ---------------------------------------------------------
print(f"\n{len(df)} runs finished")
if len(df):
    print(df.groupby(["fraction", "method"])[["dev_f1_macro", "test_f1_macro"]]
            .agg(["mean", "std", "count"]).round(4).to_string())


### Cell 8 - one small zip with everything the analysis needs

Manifests, curves, predictions, figures. The big full-FT model files are not
included (they take ~11 GB and the thesis doesn't use them). Downloads the zip
to the browser AND leaves a copy on Drive.

In [ ]:
# ---- cell 8: one small zip with everything the analysis needs ----
# WHAT: bundle runs/, curves_v2/, preds/, figures/ + top-level CSVs into one zip.
# WHY:  a single self-contained artifact to copy to another machine, share, or
#       import on another machine. Nothing is overwritten on re-import.

INCOMING = ""   # e.g. f"{PROJECT_DIR}/thesis_runs_5060_20260901_2210.zip"
if INCOMING:
    import_zip(INCOMING)

import shutil, os
STAGE = "/content/step10_artifacts"
shutil.rmtree(STAGE, ignore_errors=True)
os.makedirs(STAGE, exist_ok=True)
for name in ["runs", "curves_v2", "preds", "figures", "runs_progress"]:
    src = f"{PROJECT_DIR}/{name}"
    if os.path.exists(src):
        shutil.copytree(src, f"{STAGE}/{name}")
for f in ["results_v2.csv", "significance.csv", "results_log.csv",
          "failures.log"]:
    if os.path.exists(f"{PROJECT_DIR}/{f}"):
        shutil.copy(f"{PROJECT_DIR}/{f}", STAGE)
zip_path = shutil.make_archive(f"{PROJECT_DIR}/step10_artifacts", "zip", STAGE)
print(zip_path, round(os.path.getsize(zip_path) / 1e6, 2), "MB")

# Download to browser as well, so a copy exists off Drive
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print(f"  browser download unavailable ({e}) - the zip is on Drive already.")


### Cell 9 - analysis: tables, significance, figures (no GPU)

Reads `runs/*.json` and `preds/*.npz` and prints, in order:

1. **Consistency checks** - anything that would make the comparison unfair.
2. **Test macro-F1** by data size and method (mean +/- sd over seeds).
3. **Paired per-seed gaps** (LoRA minus full FT at each seed).
4. **Significance**: McNemar exact test on accuracy + paired bootstrap CI on
   macro-F1.
5. **Efficiency table**: params, wall-clock, memory, model size.
6. **Figures** into `figures/`.

Read the consistency block first. If it reports a ceiling still binding, seeds
that differ between methods, or missing predictions, resolve those before
reading the numbers.

In [ ]:
# ---- cell 9: analysis, tables, significance, figures ----
# =============================================================================
# STEP 11 - analysis: test-set tables, significance, efficiency, figures
# =============================================================================
# Reads Step 10's outputs (runs/*.json + preds/*.npz). No GPU, no training.
# Same logic as the v1 analysis cell, so numbers stay comparable across notebooks.
# The README's full statistics (TOST, Holm, ECE, temperature scaling) come from
# scripts/step12_paper_stats.py, which runs on the exported artifacts.
# =============================================================================

import glob, itertools, json, os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from scipy.stats import binomtest

RUNS_DIR   = f"{PROJECT_DIR}/runs"
PREDS_DIR  = f"{PROJECT_DIR}/preds"
FIG_DIR    = f"{PROJECT_DIR}/figures"
OLD_LOG    = f"{PROJECT_DIR}/results_log.csv"
BOOT_N     = 2000
os.makedirs(FIG_DIR, exist_ok=True)

runs = [json.load(open(p)) for p in sorted(glob.glob(f"{RUNS_DIR}/*.json"))]
if not runs:
    raise SystemExit("no runs/*.json yet - run Step 10 first")
df = pd.DataFrame(runs).sort_values(["fraction", "method", "seed"]).reset_index(drop=True)
print(f"{len(df)} runs: "
      + ", ".join(f"{f:g}={n}" for f, n in df.groupby('fraction').size().items()))

# ---- 1. consistency checks -------------------------------------------------
print("\n=== consistency ===")
problems = []
for col in ["precision", "batch_size", "epoch_ceiling", "patience_epochs"]:
    if col in df.columns and df[col].nunique(dropna=False) > 1:
        problems.append(f"{col} differs across runs: "
                        f"{sorted(df[col].unique(), key=str)}")
for frac, g in df.groupby("fraction"):
    for col in ["patience_evals", "evals_per_epoch"]:
        if col in g.columns and g[col].nunique(dropna=False) > 1:
            problems.append(f"fraction {frac:g}: {col} differs within fraction")
for (frac, method), g in df.groupby(["fraction", "method"]):
    if g["seed"].duplicated().any():
        problems.append(f"duplicate seeds at fraction {frac}, {method}")
for frac, g in df.groupby("fraction"):
    per = g.groupby("method")["seed"].apply(set)
    if len(per) < 2:
        problems.append(f"fraction {frac}: only {list(per.index)} - no comparison")
    elif len(per) == 2 and per.iloc[0] != per.iloc[1]:
        problems.append(f"fraction {frac}: seeds differ between methods")
missing_preds = [r for r in df["run_id"]
                 if not os.path.exists(f"{PREDS_DIR}/{r}_test.npz")]
if missing_preds:
    problems.append(f"no saved predictions for {missing_preds}")
hit = df[df["ceiling_hit"]]
if len(hit):
    problems.append(f"ceiling still binding for {list(hit['run_id'])}")
drift = df[df["dev_reload_drift"] > 1e-3]
if len(drift):
    problems.append(f"best model did not reload cleanly for {list(drift['run_id'])}")
# [v2] reload verification check
if "reload_verified" in df.columns:
    unver = df[~df["reload_verified"].fillna(True).astype(bool)]
    if len(unver):
        problems.append(f"reload verification failed for {list(unver['run_id'])}")
    big_drift = df[(df.get("reload_max_logit_drift").fillna(0) > 1e-3)]
    if len(big_drift):
        problems.append(f"logit drift >1e-3 on reload for {list(big_drift['run_id'])}")

MACHINE = [c for c in ("gpu", "torch", "transformers", "peft") if c in df.columns]
multi_machine = {c: sorted(df[c].dropna().unique()) for c in MACHINE
                 if df[c].nunique(dropna=False) > 1}
for (frac, seed), g in df.groupby(["fraction", "seed"]):
    for c in multi_machine:
        if g[c].nunique(dropna=False) > 1:
            problems.append(f"fraction {frac:g} seed {seed} split across "
                            f"{sorted(g[c].dropna().unique(), key=str)} ({c})")
print("\n".join(f"  [!] {p}" for p in problems) if problems else "  nothing to flag")
if multi_machine:
    for c, vals in multi_machine.items():
        print(f"  note: {c} varies: {vals}")

rule = df["patience_epochs"].dropna().unique()
print("  stopping rule: " + (f"early stop patience {rule[0]:g}" if len(rule)
      else f"none - {df['epoch_ceiling'].max():g} full epochs, best on val"))
print(f"  early stopped: {int(df['early_stopped'].sum())}/{len(df)} "
      f"({dict(df.groupby('method')['early_stopped'].sum())})")

# ---- 2. test scores by fraction and method --------------------------------
print("\n=== TEST macro-F1 (mean +/- sd over seeds) ===")
agg = (df.groupby(["fraction", "method"])[["test_f1_macro", "test_accuracy"]]
         .agg(["mean", "std", "count"]).round(4))
print(agg.to_string())

# ---- 3. paired per-seed gaps ----------------------------------------------
print("\n=== paired gaps, LoRA minus full FT (per seed) ===")
wide = df.pivot_table(index=["fraction", "seed"], columns="method",
                      values="test_f1_macro")
pairs = wide.dropna(subset=[c for c in ("lora", "full_ft") if c in wide.columns])
if {"lora", "full_ft"}.issubset(pairs.columns):
    pairs = pairs.assign(gap=(pairs["lora"] - pairs["full_ft"]).round(4))
    print(pairs.round(4).to_string())
    summary = pairs.groupby("fraction")["gap"].agg(
        ["mean", "std", "count", lambda s: int((s > 0).sum())])
    summary.columns = ["mean_gap", "sd", "n_seeds", "seeds_lora_ahead"]
    print("\n" + summary.round(4).to_string())
else:
    print("  need both methods at the same fraction+seed to pair")

# ---- 4. significance ------------------------------------------------------
def load_preds(run_id):
    z = np.load(f"{PREDS_DIR}/{run_id}_test.npz")
    return z["logits"].astype(np.float32).argmax(1), z["labels"]

def mcnemar(a_pred, b_pred, y):
    a, b_ = (a_pred == y), (b_pred == y)
    b = int((a & ~b_).sum())
    c = int((~a & b_).sum())
    p = binomtest(b, b + c, 0.5).pvalue if (b + c) else 1.0
    return b, c, p

def boot_f1_gap(a_pred, b_pred, y, n=BOOT_N, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(y), size=(n, len(y)))
    d = np.array([f1_score(y[i], a_pred[i], average="macro")
                  - f1_score(y[i], b_pred[i], average="macro") for i in idx])
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

print("\n=== significance, per fraction and seed (LoRA vs full FT) ===")
print("  b = LoRA right / full FT wrong; c = reverse; p = exact McNemar.")
sig_rows = []
for (frac, seed), g in df.groupby(["fraction", "seed"]):
    have = dict(zip(g["method"], g["run_id"]))
    if not {"lora", "full_ft"}.issubset(have): continue
    if any(not os.path.exists(f"{PREDS_DIR}/{have[m]}_test.npz")
           for m in ("lora", "full_ft")): continue
    lp, y = load_preds(have["lora"])
    fp, y2 = load_preds(have["full_ft"])
    assert np.array_equal(y, y2), "test labels differ - wrong preds file"
    b, c, p = mcnemar(lp, fp, y)
    f1_l = f1_score(y, lp, average="macro")
    f1_f = f1_score(y, fp, average="macro")
    lo, hi = boot_f1_gap(lp, fp, y)
    sig_rows.append({"fraction": frac, "seed": seed,
                     "f1_lora": round(f1_l, 4), "f1_full_ft": round(f1_f, 4),
                     "f1_gap": round(f1_l - f1_f, 4),
                     "gap_ci95": f"[{lo:+.4f}, {hi:+.4f}]",
                     "ci_excludes_0": (lo > 0) or (hi < 0),
                     "mcnemar_b": b, "mcnemar_c": c,
                     "mcnemar_p": round(p, 4), "sig_05": p < 0.05})
sig = pd.DataFrame(sig_rows)
if len(sig):
    print(sig.to_string(index=False))
    sig.to_csv(f"{PROJECT_DIR}/significance.csv", index=False)
    n_sig = int(sig["sig_05"].sum())
    print(f"\n  {n_sig} of {len(sig)} comparisons reach p<0.05 on accuracy; "
          f"{int(sig['ci_excludes_0'].sum())} have F1 CI excluding zero.")
else:
    print("  no paired prediction files yet")

# ---- 5. efficiency --------------------------------------------------------
print("\n=== efficiency (what LoRA is actually for) ===")
if "wall_clock_comparable" in df.columns:
    df["wall_clock_comparable"] = df["wall_clock_comparable"].fillna(True).astype(bool)
timed = df[df["wall_clock_comparable"]] if "wall_clock_comparable" in df.columns else df
if "wall_clock_comparable" in df.columns and len(timed) < len(df):
    print(f"  {len(df) - len(timed)} resumed run(s) excluded from timing")
by = ["method", "gpu"] if ("gpu" in df.columns and df["gpu"].nunique() > 1) \
     else ["method"]
eff = (df.groupby(by)
         .agg(runs=("run_id", "count"),
              trainable_params=("trainable_params", "max"),
              pct_trainable=("pct_trainable", "max"),
              saved_model_mb=("saved_model_mb", "mean"),
              peak_gpu_mem_mb=("peak_gpu_mem_mb", "mean")).round(2))
eff = eff.join(timed.groupby(by).agg(
    timed_runs=("run_id", "count"),
    samples_per_s=("train_samples_per_s", "mean")).round(2))
print(eff.to_string())
print("\nwall-clock minutes per run, by fraction:")
idx = ["gpu", "fraction"] if len(by) == 2 else "fraction"
print((timed.pivot_table(index=idx, columns="method",
                         values="train_runtime_s", aggfunc="mean") / 60)
      .round(1).to_string())

# ---- 6. figures -----------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for method, style in (("full_ft", "o-"), ("lora", "s--")):
    g = df[df["method"] == method].groupby("fraction")["test_f1_macro"]
    if not len(g): continue
    ax.errorbar(g.mean().index, g.mean().values,
                yerr=g.std().fillna(0).values, fmt=style, capsize=3,
                label="full fine-tuning" if method == "full_ft" else "LoRA")
ax.set_xscale("log")
ax.set_xlabel("fraction of training data (log scale)")
ax.set_ylabel("test macro-F1")
ax.set_title("Matched epoch budget, selection on validation, reported on test")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(f"{FIG_DIR}/test_f1_vs_fraction.{ext}", dpi=200)
print(f"\nwrote {FIG_DIR}/test_f1_vs_fraction.png/.pdf")

fig2, ax2 = plt.subplots(figsize=(6, 4))
for _, r in df.iterrows():
    cp = f"{PROJECT_DIR}/curves_v2/{r['run_id']}.json"
    if not os.path.exists(cp): continue
    c = json.load(open(cp))
    ax2.plot([e["epoch"] for e in c], [e["eval_f1_macro"] for e in c],
             ("s--" if r["method"] == "lora" else "o-"), alpha=0.7,
             color=("tab:orange" if r["method"] == "lora" else "tab:blue"),
             label=r["method"] if r["run_id"].endswith(str(df["seed"].iloc[0])) else None)
ax2.set_xlabel("epoch"); ax2.set_ylabel("validation macro-F1")
ax2.set_title("Convergence at a matched ceiling")
ax2.grid(alpha=0.3)
handles, labels = ax2.get_legend_handles_labels()
if labels:
    ax2.legend(dict(zip(labels, handles)).values(),
               dict(zip(labels, handles)).keys())
fig2.tight_layout()
for ext in ("png", "pdf"):
    fig2.savefig(f"{FIG_DIR}/dev_curves_matched.{ext}", dpi=200)
print(f"wrote {FIG_DIR}/dev_curves_matched.png/.pdf")

if os.path.exists(OLD_LOG):
    old = pd.read_csv(OLD_LOG)
    keep = old[~old["fraction"].isin(df["fraction"].unique())]
    if len(keep):
        print("\n=== first pass, DEV only (label as dev, unmatched budgets) ===")
        print(keep.pivot_table(index="fraction", columns="method",
                               values="f1_macro", aggfunc="mean").round(4).to_string())


### Reading the output

- **Accuracy.** Read the per-seed gaps and the significance test together. A
  confidence interval that includes zero with p > 0.05 means no measurable
  difference at that data size, even if the mean gap is positive. Equivalence
  needs a separate test (TOST) against a stated margin; see
  `scripts/step12_paper_stats.py`.
- **Efficiency.** Trainable parameters and artefact size are hardware-independent.
  Wall-clock and peak memory are only comparable within one GPU; keep the GPU
  column when summarising.
- **Wall-clock.** Report time to the selected checkpoint as well as the full
  8-epoch time. At 5% data full fine-tuning peaks around epoch 2 and LoRA around
  epoch 7, so on an A100 LoRA reaches its kept model later.
- **Reload verification.** `reload_max_logit_drift` is recorded for every run
  made with this notebook (the 16 A100 runs).
- **Failed runs.** Any run that collapsed below `CHANCE_F1` has no manifest and
  appears in `failures.log`; report the count and cause.
